
Import Library


In [2]:
import os, json, random, zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from huggingface_hub import HfApi, hf_hub_download

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


config

In [3]:
CONFIG = {
    "image_data_dir": Path("/content/data/hf_download"),   # contains train/<crop>/, val/<crop>/, test/<crop>/
    "labels_registry": Path("/content/data/hf_download/labels_registry.jsonl"),

    "models_dir": Path("/content/models"),
    "results_dir": Path("/content/results"),

    "img_size": 128,
    "cnn_epochs": 15,
    "cnn_lr": 1e-3,
    "cnn_batch_size": 32,

    "seed": SEED,
}
CONFIG["models_dir"].mkdir(parents=True, exist_ok=True)
CONFIG["results_dir"].mkdir(parents=True, exist_ok=True)

download from hugging face and unzip

In [4]:


HF_EXTRACT_TO = Path("/content/data/hf_download")
HF_EXTRACT_TO.mkdir(parents=True, exist_ok=True)

local_zip = hf_hub_download(
    repo_id="w4ashabii/nepali_crop_data",
    repo_type="dataset",
    filename="zips/dataset_part_000.zip",
)
with zipfile.ZipFile(local_zip) as zf:
    zf.extractall(HF_EXTRACT_TO)

CONFIG["image_data_dir"] = HF_EXTRACT_TO
CONFIG["data_path"] = HF_EXTRACT_TO / "sharegpt.jsonl"

print("image_data_dir:", CONFIG["image_data_dir"])
print("data_path:", CONFIG["data_path"])
!find {HF_EXTRACT_TO} -maxdepth 2 -type d

zips/dataset_part_000.zip: reconstructing file:   0%|          |  0.00B / 1.52GB            

zips/dataset_part_000.zip: downloading bytes:           |  0.00B            

image_data_dir: /content/data/hf_download
data_path: /content/data/hf_download/sharegpt.jsonl
/content/data/hf_download
/content/data/hf_download/train
/content/data/hf_download/train/pepper
/content/data/hf_download/train/strawberry
/content/data/hf_download/train/potato
/content/data/hf_download/train/cherry
/content/data/hf_download/train/maize
/content/data/hf_download/train/apple
/content/data/hf_download/train/peach
/content/data/hf_download/train/tomato
/content/data/hf_download/train/grape
/content/data/hf_download/val
/content/data/hf_download/val/pepper
/content/data/hf_download/val/strawberry
/content/data/hf_download/val/potato
/content/data/hf_download/val/maize
/content/data/hf_download/val/apple
/content/data/hf_download/val/peach
/content/data/hf_download/val/tomato
/content/data/hf_download/val/grape
/content/data/hf_download/test
/content/data/hf_download/test/potato
/content/data/hf_download/test/maize
/content/data/hf_download/test/apple
/content/data/hf_download/te

In [5]:
train_tf = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_img_ds = datasets.ImageFolder(CONFIG["image_data_dir"] / "train", transform=train_tf)
val_img_ds = datasets.ImageFolder(CONFIG["image_data_dir"] / "val", transform=val_tf)

train_loader = DataLoader(train_img_ds, batch_size=CONFIG["cnn_batch_size"], shuffle=True, num_workers=2)
val_loader = DataLoader(val_img_ds, batch_size=CONFIG["cnn_batch_size"], shuffle=False, num_workers=2)

NUM_CLASSES = len(train_img_ds.classes)
print("Classes:", train_img_ds.classes)

Classes: ['apple', 'cherry', 'grape', 'maize', 'peach', 'pepper', 'potato', 'strawberry', 'tomato']


Train cnn

In [6]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)

        feat_size = CONFIG["img_size"] // 8
        self.fc1 = nn.Linear(128 * feat_size * feat_size, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x


cnn_baseline = BaselineCNN(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.Adam(cnn_baseline.parameters(), lr=CONFIG["cnn_lr"], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["cnn_epochs"])
criterion = nn.CrossEntropyLoss()

In [7]:
cnn_history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0
cnn_final_dir = CONFIG["models_dir"] / "baseline_cnn_final"
cnn_final_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(CONFIG["cnn_epochs"]):
    cnn_baseline.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = cnn_baseline(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    cnn_baseline.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = cnn_baseline(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    scheduler.step()

    cnn_history["train_loss"].append(train_loss)
    cnn_history["val_loss"].append(val_loss)
    cnn_history["train_acc"].append(train_acc)
    cnn_history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{CONFIG['cnn_epochs']} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(cnn_baseline.state_dict(), cnn_final_dir / "baseline_cnn.pt")

json.dump(cnn_history, open(CONFIG["results_dir"] / "baseline_cnn_history.json", "w"))
print(f"Best val_acc: {best_val_acc:.4f} | saved to {cnn_final_dir}")

Epoch 1/15 | train_loss=1.1548 train_acc=0.6309 | val_loss=8.3776 val_acc=0.0868
Epoch 2/15 | train_loss=0.6559 train_acc=0.7769 | val_loss=9.9322 val_acc=0.1246
Epoch 3/15 | train_loss=0.5028 train_acc=0.8313 | val_loss=11.2351 val_acc=0.1335
Epoch 4/15 | train_loss=0.4001 train_acc=0.8665 | val_loss=13.1364 val_acc=0.1414
Epoch 5/15 | train_loss=0.3447 train_acc=0.8881 | val_loss=11.6094 val_acc=0.1503
Epoch 6/15 | train_loss=0.2992 train_acc=0.9012 | val_loss=14.5053 val_acc=0.1328
Epoch 7/15 | train_loss=0.2659 train_acc=0.9121 | val_loss=14.4551 val_acc=0.1460
Epoch 8/15 | train_loss=0.2266 train_acc=0.9246 | val_loss=13.2415 val_acc=0.1491
Epoch 9/15 | train_loss=0.1985 train_acc=0.9358 | val_loss=14.4736 val_acc=0.1544
Epoch 10/15 | train_loss=0.1689 train_acc=0.9446 | val_loss=14.7748 val_acc=0.1486
Epoch 11/15 | train_loss=0.1407 train_acc=0.9548 | val_loss=16.4154 val_acc=0.1505
Epoch 12/15 | train_loss=0.1210 train_acc=0.9604 | val_loss=17.0110 val_acc=0.1515
Epoch 13/15 | t